# The HTTP API, hooks and the inbox

`amoorion api` serves every method at `POST /methods/<name>` and incoming
webhooks at `POST /hooks/<name>`. Here the app runs in-process through
FastAPI's test client, so nothing listens on a port.

In [1]:
import os

from amoorion import Amoorion

# A sandbox for this whole notebook: secrets only from variables, the inbox and
# the dataset in a temporary folder. Nothing here can touch your real setup.
Amoorion.init()

# The API's own token and the generic hook's token, as the vault would hold them.
# Tokens for the API and for a hook, in the sandbox's variables.
os.environ["AMOORION_SECRET_API_TOKEN"] = "api-example-token"

os.environ["AMOORION_SECRET_HOOKS_GENERIC_TOKEN"] = "hook-example-token"

In [2]:
# Calls the application directly: no port is opened.
from fastapi.testclient import TestClient

from amoorion.api import create_app

client = TestClient(create_app())

# The open route: what it serves.
client.get("/").json()

{'name': 'amoorion',
 'version': '0.4.32a0',
 'methods': ['accounts',
  'compose',
  'delete',
  'inbox',
  'notify',
  'plugins',
  'publish',
  'read',
  'reply',
  'search',
  'simulate',
  'skill',
  'snapshot',
  'wiki',
  'workflow'],
 'hooks': ['discord', 'generic', 'github', 'gitlab', 'plane', 'sentry']}

Method calls need the API token as a bearer token:

In [3]:
# 401 without a token.
print(
    client.post(
        "/methods/publish",
        json={
            "text": "x",
            "to": ["dryrun"],
        },
    ).status_code
)

# With one, it goes through.
auth = {"Authorization": "Bearer api-example-token"}

client.post(
    "/methods/publish",
    json={
        "text": "posted over HTTP",
        "to": ["dryrun"],
    },
    headers=auth,
).json()

401


[{'platform': 'dryrun',
  'ok': True,
  'message': {'platform': 'dryrun',
   'text': 'posted over HTTP',
   'id': '1',
   'url': None,
   'author': None,
   'reply_to': None,
   'created_at': '2026-10-31T07:55:17.268775Z'},
  'error': None}]

A service tells amoorion that something happened through a hook. The
`generic` hook takes `{"text", "author"}` with its token in
`X-Amoorion-Token`; what it receives goes to the inbox:

In [4]:
# A hook has its own token, so a sender can do only this.
client.post(
    "/hooks/generic",
    json={
        "text": "deploy finished",
        "author": "ci",
    },
    headers={"X-Amoorion-Token": "hook-example-token"},
).json()

{'platform': 'hook:generic',
 'text': 'deploy finished',
 'id': '324175d9179045f09cf3b0e7541fc0ce',
 'url': None,
 'author': {'platform': 'hook:generic', 'handle': 'ci', 'name': None},
 'reply_to': None,
 'created_at': None}

In [5]:
# Both came to the same inbox.
client.post(
    "/methods/inbox",
    json={},
    headers=auth,
).json()

[{'platform': 'hook:generic',
  'text': 'deploy finished',
  'id': '324175d9179045f09cf3b0e7541fc0ce',
  'url': None,
  'author': {'platform': 'hook:generic', 'handle': 'ci', 'name': None},
  'reply_to': None,
  'created_at': None}]